In [1]:
import pandas as pd 
import numpy as np 
import re

In [2]:
raw=pd.read_excel("project raw file.xlsx")
raw

,stNo,EmpName,Unnamed: 2
0,ST2462,Theodora Kansiime,NaN
1,ST1917,Oola Hilda,NaN
2,ST2208,Esebu Edward,NaN
3,ST2735,Muwumba Norah,NaN
4,ST1787,Kababiito Winfred,NaN
...,...,...,...
340,ST2545,Shallot Atukunda,NaN
341,ST2419,Ileka Grace,NaN
342,ST2446,Emuria Joseph,NaN
343,ST1853,Mugisa Michael,NaN


In [8]:
# Clean spaces
raw["EmpName"] = (
    raw["EmpName"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [ ]:
def standardize_name(name):
    parts = name.split()

    # One name
    if len(parts) == 1:
        return parts[0].title()

    # Two names
    elif len(parts) == 2:
        first, second = parts

        # Assume surname often comes first in such cases
        return f"{second.title()} {first.title()}"

    # Three names
    elif len(parts) == 3:
        a, b, c = parts
    #Four names
    elif len(parts)== 4:
        a,b,c,d =parts

        # Put surname last
        return f"{b.title()} {c.title()} {d.title()} {a.title()}"

    # Four or more names
    else:
        surname = parts[0]
        others = parts[1:]

        return " ".join([x.title() for x in others] + [surname.title()])

In [10]:
# Apply standardization
raw["Standard_Name"] = raw["EmpName"].apply(standardize_name)

# Build a single standard name per ID
id_lookup = (
    raw.groupby("stNo")["Standard_Name"]
      .agg(lambda x: x.value_counts().index[0])
      .to_dict()
)


In [ ]:
# Money cleaning function (for future files)
def standardize_money(value):
    if pd.isna(value):
        return value

    text = str(value).lower().replace(',', '').strip()
    text = text.replace('ugx', '').strip()

    multipliers = {
        'k': 1_000,
        'thousand': 1_000,
        'm': 1_000_000,
        'million': 1_000_000,
        'b': 1_000_000_000,
        'billion': 1_000_000_000,
    }

    match = re.match(r'(\d+(?:\.\d+)?)\s*([a-z]+)?', text)
    if match:
        num = float(match.group(1))
        suffix = match.group(2)

        if suffix in multipliers:
            num *= multipliers[suffix]

        return int(num)

    return value

In [11]:
OUTPUT_FILE = "cleaned_project_file.xlsx"
# Replace all names for the same ID
raw["Standard_Name"] = raw["stNo"].map(id_lookup)

# Save output
raw.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")

print(f"Saved: {OUTPUT_FILE}")

Saved: cleaned_project_file.xlsx
